# LLM Sliding Window Reranker

Simple sliding-window reranker over validation founders using Gemini 2.5 Pro.

**Validation-only inputs:**
- `solution_sets/@ datasets/prediction_data/val_predictions.csv`
- `solution_sets/@ datasets/training/founder_experience_training.csv`
- `.env` in this folder with `GEMINI_API_KEY`.


## 1. Install dependencies

One-time lightweight install of libraries used in this notebook.

In [1]:
%pip install -q google-generativeai python-dotenv jinja2 pandas numpy tqdm


Note: you may need to restart the kernel to use updated packages.


## 2. Imports and global configuration

Define all global variables (hyperparameters and paths) at the top and resolve the repo root.

In [2]:
import os
from pathlib import Path
import json
import re

import numpy as np
import pandas as pd
from dotenv import load_dotenv
import google.generativeai as genai
from jinja2 import Template
from tqdm.auto import tqdm

# Global config
WINDOW_SIZE = 20           # sliding window size
RERANK_TOP_N = 50          # only rerank top N founders
K_METRIC = 20              # top-k for recall / NDCG
GEMINI_MODEL = "gemini-2.5-flash"

def get_repo_root() -> Path:
    """Return repo root (directory that contains solution_sets and AGENTS.md)."""
    p = Path.cwd().resolve()
    while True:
        if (p / "solution_sets").exists() and (p / "AGENTS.md").exists():
            return p
        if p.parent == p:
            return Path.cwd().resolve()
        p = p.parent

REPO_ROOT = get_repo_root()

# Paths (rooted at repo)
PREDICTIONS_PATH = REPO_ROOT / "solution_sets" / "@ datasets" / "prediction_data" / "val_predictions.csv"
EXPERIENCE_PATH = REPO_ROOT / "solution_sets" / "@ datasets" / "training" / "founder_experience_training.csv"
ENV_PATH = REPO_ROOT / "solution_sets" / "@ simple_benchmarks" / "llm_sliding_window_ranker" / ".env"
TEMPLATE_PATH = REPO_ROOT / "solution_sets" / "@ simple_benchmarks" / "llm_sliding_window_ranker" / "propmt.j2"

assert PREDICTIONS_PATH.exists(), f"Missing predictions file: {PREDICTIONS_PATH}"
assert EXPERIENCE_PATH.exists(), f"Missing experience file: {EXPERIENCE_PATH}"
assert ENV_PATH.exists(), f"Missing .env file: {ENV_PATH}"
assert TEMPLATE_PATH.exists(), f"Missing prompt template file: {TEMPLATE_PATH}"


/usr/local/google/home/guyu/Desktop/gv_case_study/submission/source/xgboost/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 3. Load data and Gemini model

Use the `.env` key to configure Gemini and load the validation predictions and experience tables.

In [3]:
# Configure Gemini from local .env
load_dotenv(ENV_PATH)
api_key = os.getenv("GEMINI_API_KEY")
if not api_key:
    raise RuntimeError("GEMINI_API_KEY not found in .env")

genai.configure(api_key=api_key)
model = genai.GenerativeModel(GEMINI_MODEL)

# Load validation data
pred_df = pd.read_csv(PREDICTIONS_PATH)
exp_df = pd.read_csv(EXPERIENCE_PATH)

# Normalise person_id key across files
exp_df["person_key"] = exp_df["person_id"].str.replace(r"^/p/", "", regex=True)
pred_df["person_key"] = pred_df["person_id"]

print("Loaded:")
print("  predictions rows:", len(pred_df))
print("  experience rows:", len(exp_df))


Loaded:
  predictions rows: 714
  experience rows: 54616


## 4. Helper functions

Build founder property strings, call the LLM for a window, apply the sliding-window reranker with `tqdm`, and compute recall / NDCG.

In [4]:
# Load Jinja prompt template
FOUNDER_PROMPT_TEMPLATE = Template(TEMPLATE_PATH.read_text(encoding="utf-8"))


def build_founder_properties(row: pd.Series, exp: pd.DataFrame) -> str:
    person_id = row["person_id"]
    experiences = exp.loc[exp["person_key"] == person_id]
    # Summarise up to a few recent roles to keep prompt short
    exp_texts = []
    for _, r in experiences.sort_values("order").tail(5).iterrows():
        company = (r.get("company_id") or "?")
        title = (r.get("title") or "?")
        job_type = (r.get("job_type") or "?")
        start = (r.get("start_date") or "?")
        end = (r.get("end_date") or "present")
        exp_texts.append(f"{title} at {company} ({job_type}, {start} to {end})")
    exp_summary = "; ".join(exp_texts) if exp_texts else "No detailed experience rows found."

    perf_mean = row.get("founder_perf_mean", np.nan)
    perf_max = row.get("founder_perf_max", np.nan)
    perf_last = row.get("founder_perf_last", np.nan)
    total_years = row.get("total_experience_years", np.nan)
    network_size = row.get("network_size_1st", np.nan)
    network_quality = row.get("network_quality_1st", np.nan)
    team_size = row.get("team_size", np.nan)
    industry = row.get("industry", "?")
    label_industry = row.get("label_company_industry", "?")
    edu_tier = row.get("education_tier", np.nan)
    edu_level = row.get("education_level_score", np.nan)
    n_founder_roles = row.get("n_founder_roles", np.nan)
    n_exec_roles = row.get("n_executive_roles", np.nan)
    n_c_suite_roles = row.get("n_c_suite_roles", np.nan)
    n_board_roles = row.get("n_board_roles", np.nan)

    text = (
        f"Career experience: total {total_years:.1f} years, "
        f"founder roles={n_founder_roles}, executive roles={n_exec_roles}, "
        f"C-suite roles={n_c_suite_roles}, board roles={n_board_roles}. "
        f"Recent roles: {exp_summary}. "
        f"Previous company performance: mean={perf_mean}, max={perf_max}, last={perf_last}. "
        f"Network: size_1st={network_size}, quality_1st={network_quality}, team_size={team_size}. "
        f"Education: tier={edu_tier}, level_score={edu_level}. "
        f"Domain: current industry={industry}, label_company_industry={label_industry}."
    )
    return text


def call_gemini_for_window(window_person_ids, rows_by_person, exp: pd.DataFrame):
    founders = []
    for pid in window_person_ids:
        row = rows_by_person[pid]
        founders.append(
            {
                "person_id": pid,
                "properties": build_founder_properties(row, exp),
            }
        )

    prompt = FOUNDER_PROMPT_TEMPLATE.render(founders=founders)
    response = model.generate_content(prompt)
    text = response.text or ""

    # Extract JSON list of person_ids from the response
    match = re.search(r"\[.*\]", text, re.DOTALL)
    if not match:
        raise ValueError(f"Model output has no JSON list: {text}")
    json_str = match.group(0)
    order = json.loads(json_str)

    # Keep only IDs from the current window and preserve any missing ones at the end
    allowed = set(window_person_ids)
    ordered = [pid for pid in order if pid in allowed]
    remaining = [pid for pid in window_person_ids if pid not in ordered]
    return ordered + remaining


def sliding_window_rerank(person_ids, rows_by_person, exp: pd.DataFrame, window_size: int) -> list:
    ids = list(person_ids)
    n = len(ids)
    if n <= window_size:
        return call_gemini_for_window(ids, rows_by_person, exp)

    # Slide from bottom up (windows overlap by window_size - 1)
    for start in tqdm(
        range(n - window_size, -1, -1), desc="Sliding windows", leave=False
    ):
        window_ids = ids[start : start + window_size]
        new_window = call_gemini_for_window(window_ids, rows_by_person, exp)
        ids[start : start + window_size] = new_window
    return ids


def dcg_at_k(labels: np.ndarray, k: int) -> float:
    labels = np.asarray(labels, dtype=float)[:k]
    if labels.size == 0:
        return 0.0
    discounts = 1.0 / np.log2(np.arange(2, labels.size + 2))
    return float(np.sum(labels * discounts))


def ndcg_at_k(labels: np.ndarray, k: int) -> float:
    ideal = np.sort(labels)[::-1]
    ideal_dcg = dcg_at_k(ideal, k)
    if ideal_dcg == 0:
        return 0.0
    return dcg_at_k(labels, k) / ideal_dcg


def recall_at_k(labels: np.ndarray, k: int) -> float:
    labels = np.asarray(labels, dtype=float)
    total_relevant = labels.sum()
    if total_relevant == 0:
        return 0.0
    top_k_relevant = labels[:k].sum()
    return float(top_k_relevant / total_relevant)

def tournament_rerank(person_ids, rows_by_person, exp, k_top=20, bracket_size=5):
    """
    Tournament sort to find top-K.
    Repeatedly runs brackets to find the best, then removes best and repeats.
    Optimized to reuse comparisons where possible is complex, so we do a simpler version:
    Run tournament for Rank 1, then Rank 2, etc., excluding already found.
    """
    ids = list(person_ids)
    top_k = []
    
    for _ in tqdm(range(k_top), desc="Tournament Rounds"):
        if not ids:
            break
        
        # If remaining ids are few, just rank them all
        if len(ids) <= bracket_size:
            ranked = call_gemini_for_window(ids, rows_by_person, exp)
            top_k.append(ranked[0])
            ids.remove(ranked[0])
            continue
        
        # Run brackets
        current_level = ids
        while len(current_level) > 1:
            next_level = []
            # Split into brackets
            for i in range(0, len(current_level), bracket_size):
                bracket = current_level[i:i+bracket_size]
                if len(bracket) == 1:
                    next_level.append(bracket[0])
                    continue
                
                ranked = call_gemini_for_window(bracket, rows_by_person, exp)
                next_level.append(ranked[0]) # Winner moves up
                # Losers stay in current_level or are eliminated from this round
            current_level = next_level
        
        winner = current_level[0]
        top_k.append(winner)
        ids.remove(winner)
    
    return top_k + ids # Return top_k followed by remaining



## 5. Run reranker and compute metrics

Apply the sliding-window reranker on the validation ranking and compare baseline vs LLM reranked Recall@K and NDCG@K.

In [ ]:
# Baseline ranking: higher predicted_score = better
baseline_df = pred_df.sort_values("predicted_score", ascending=False).reset_index(drop=True)
baseline_ids = baseline_df["person_id"].tolist()

# Map from person_id to row for quick lookup
rows_by_person = {row["person_id"]: row for _, row in baseline_df.iterrows()}

# Only rerank the top RERANK_TOP_N founders to save LLM calls
top_ids = baseline_ids[:RERANK_TOP_N]
rest_ids = baseline_ids[RERANK_TOP_N:]

print(f"Running tournament reranker for top {K_METRIC} out of {RERANK_TOP_N}")
reranked_top_ids = tournament_rerank(top_ids, rows_by_person, exp_df, k_top=K_METRIC, bracket_size=5)
reranked_ids = reranked_top_ids + rest_ids

# Metrics (single global ranking)
labels_all = baseline_df["label"].to_numpy()
baseline_recall = recall_at_k(labels_all, K_METRIC)
baseline_ndcg = ndcg_at_k(labels_all, K_METRIC)

reranked_labels = baseline_df.set_index("person_id").loc[reranked_ids]["label"].to_numpy()
reranked_recall = recall_at_k(reranked_labels, K_METRIC)
reranked_ndcg = ndcg_at_k(reranked_labels, K_METRIC)

print(f"Baseline Recall@{K_METRIC}: {baseline_recall:.4f}")
print(f"Baseline NDCG@{K_METRIC}:   {baseline_ndcg:.4f}")
print(f"Reranked Recall@{K_METRIC}: {reranked_recall:.4f}")
print(f"Reranked NDCG@{K_METRIC}:   {reranked_ndcg:.4f}")

# Show top-20 founders before and after reranking
print("\nTop 20 person_id before reranking:")
print(baseline_ids[:K_METRIC])
print("\nTop 20 person_id after reranking:")
print(reranked_ids[:K_METRIC])


Running tournament reranker for top 20 out of 50


Tournament Rounds:  40%|████      | 8/20 [35:35<52:44, 263.71s/it]  